# 02 · Gaussian Process Regression

GPR is a powerful Bayesian non-parametric method that:

* **Quantifies uncertainty** — gives a predictive distribution, not just a point estimate
* **Works well with small datasets** — our ~2000 points are manageable
* **Incorporates prior knowledge** via kernel choice

This is especially valuable in physics: the uncertainty band on the low-x
extrapolation tells us how much we should trust the prediction in the unconstrained region.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams["figure.dpi"] = 120

from data_loader import load_lepton_dis, add_log_features, split_data, get_Xy, low_x_grid
from visualization import plot_coverage, plot_F2_vs_x, comparison_figure

DATA_DIR = "/Users/dikgarg/Desktop/Research/Neutrinos/postdoc/2025/ML/Data/Exp_data/LeptonDIS"

df_raw = load_lepton_dis(DATA_DIR)
df     = add_log_features(df_raw)
df_train, df_test = split_data(df, test_size=0.2, seed=42)

X_train, y_train = get_Xy(df_train)
X_test,  y_test  = get_Xy(df_test)

print(f"Training points: {len(X_train)}  |  Test points: {len(X_test)}")


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, WhiteKernel, ConstantKernel as C
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)


## GPR with RBF Kernel

In [ ]:
kernel_rbf = C(1.0, (1e-3, 1e3)) * RBF([1.0, 1.0], (1e-2, 1e2))              + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-4, 1.0))

gpr_rbf = GaussianProcessRegressor(
    kernel=kernel_rbf,
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=42,
)
gpr_rbf.fit(X_train_s, y_train)

y_pred_rbf, y_std_rbf = gpr_rbf.predict(X_test_s, return_std=True)

print("GPR – RBF kernel")
print(f"  Optimised kernel: {gpr_rbf.kernel_}")
print(f"  Test MSE : {mean_squared_error(y_test, y_pred_rbf):.5f}")
print(f"  Test MAE : {mean_absolute_error(y_test, y_pred_rbf):.5f}")
print(f"  Test R2  : {r2_score(y_test, y_pred_rbf):.4f}")


## GPR with Matérn Kernel (nu=3/2)

In [ ]:
kernel_mat = C(1.0, (1e-3, 1e3)) * Matern(length_scale=[1.0, 1.0],
                                                   length_scale_bounds=(1e-2, 1e2),
                                                   nu=1.5)              + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-4, 1.0))

gpr_mat = GaussianProcessRegressor(
    kernel=kernel_mat,
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=42,
)
gpr_mat.fit(X_train_s, y_train)

y_pred_mat, y_std_mat = gpr_mat.predict(X_test_s, return_std=True)

print("GPR – Matern kernel")
print(f"  Optimised kernel: {gpr_mat.kernel_}")
print(f"  Test MSE : {mean_squared_error(y_test, y_pred_mat):.5f}")
print(f"  Test MAE : {mean_absolute_error(y_test, y_pred_mat):.5f}")
print(f"  Test R2  : {r2_score(y_test, y_pred_mat):.4f}")


## Predictions with Uncertainty Bands

In [ ]:
Q2_plot = [1.0, 5.0, 15.0, 30.0]
grids   = low_x_grid(x_min=1e-6, x_max=0.8, n_points=300, Q2_values=Q2_plot)

preds_rbf = {"label": "GPR-RBF", "color": "#e41a1c", "x_arr": None,
             "Q2_preds": {}, "Q2_lo": {}, "Q2_hi": {}}
preds_mat = {"label": "GPR-Matern", "color": "#377eb8", "x_arr": None,
             "Q2_preds": {}, "Q2_lo": {}, "Q2_hi": {}}

for Q2v, (x_arr, X_feat) in grids.items():
    X_feat_s = scaler.transform(X_feat)

    mu_r, sig_r = gpr_rbf.predict(X_feat_s, return_std=True)
    mu_m, sig_m = gpr_mat.predict(X_feat_s, return_std=True)

    preds_rbf["x_arr"]           = x_arr
    preds_rbf["Q2_preds"][Q2v]   = mu_r
    preds_rbf["Q2_lo"][Q2v]      = mu_r - 2 * sig_r
    preds_rbf["Q2_hi"][Q2v]      = mu_r + 2 * sig_r

    preds_mat["x_arr"]           = x_arr
    preds_mat["Q2_preds"][Q2v]   = mu_m
    preds_mat["Q2_lo"][Q2v]      = mu_m - 2 * sig_m
    preds_mat["Q2_hi"][Q2v]      = mu_m + 2 * sig_m

fig, _ = comparison_figure(Q2_plot, df, [preds_rbf, preds_mat])
fig.suptitle("GPR predictions with 2σ uncertainty bands", y=1.01)
plt.savefig("../results/figures/02_gpr_predictions.png", dpi=150, bbox_inches="tight")
plt.show()


## Uncertainty Grows in the Low-x Extrapolation Region

The shaded bands widen as x decreases beyond the training data boundary — exactly what a
well-calibrated model should do. The RBF and Matérn kernels give similar central predictions
but differ slightly in the rate of uncertainty growth, reflecting their different smoothness assumptions.